In [ ]:
%load_ext autoreload
%autoreload 2
from dotenv import load_dotenv
from constant import *
from GeminiModel import GeminiModel
from TrainStrategy import TrainStrategy
from SimpleOutputLabelConverter import SimpleOutputLabelConverter

In [ ]:
load_dotenv()
classification_label_set = set(classify_train_dataset['label']) | set(classify_test_dataset['label'])
output_label_converter = SimpleOutputLabelConverter(classification_label_set, DEFAULT_CLASSIFICATION_CLASS)

In [ ]:
prompt_template = PromptTemplate(
    name="Manually Crafted Classification",
    definition="""
You are a experienced software engineer reviewing code comments to classify test code comments that indicate Self-Admitted Technical Debt (SATD).
A comment is considered SATD if a developer explicitly acknowledges that the code requires future work. Your task is to classify SATD comments into one of the 15 predefined categories below.

Categories of Self-Admitted Technical Debt (SATD):

Build : Issues related to the build process, such as poorly defined buid environment.

Code: Refers to poor naming conventions, unhandled or ignored exceptions, and the use of inefficient or slow algorithms.

Defect: Refers to comments that exclusively identify unresolved known defects, test case failures, or unexpected results, without referencing other types of technical debt.

Design: Refers to practices that violate the principles of good object-oriented design, such as high coupling and low cohesion.

Dependency: Refers to situations where test code relies on external dependencies, such as libraries, APIs, or services, that hinder the execution of tests due to insufficient support or incomplete functionality.

Documentation: Documentation Debt refers to the lack, incompleteness, or outdated state of code documentation.

How-To: This debt arises when there is uncertainty or a lack of clarity about how to implement, test, or resolve an issue in the code. This type of debt is characterized by unanswered questions, ambiguous comments, or speculative reasoning about expected behavior.

Impractical-Case: Refers to unexpected or problematic situations that are not anticipated under normal circumstances, yet are explicitly mentioned in comments.

Refactor: Arises when code requires restructuring, including code duplication, refactoring, cleanup, and the removal of unnecessary code. This type of debt often exists independently and does not rely on the resolution of other technical debt.

Multi: Refers to a form of technical debt that combines multiple types of debt.

Requirement: Occurs when test cases are incomplete or lack proper implementation. This debt often indicates that something needs to be done without providing specific details about the task.

Skip-Test: This debt arises when certain tests are skipped, disabled.

Subset-Test: A subset test is a special form of Skip-Test where only a small, representative portion of the full input space is used to verify the functionality, often for the sake of time efficiency or resource constraints.

Superficial-Test: This debt indicates partial or inadequate coverage of testing.

Temporary-Fix: A temporary solution implemented to address an issue, intended to be replaced with a permanent fix. It is often implicitly or explicitly mentioned as temporary due to dependencies on other issue.
    """,
    instruction="Classify the following test code comment into one of the above 15 categories. Output only the label corresponding to the most suitable category.",
    n_shot_template='Comment: "{{ text }}"',
    n_shot_answer_template= "Answer: {{ cot }} The answer is **{{ label }}**.",
    line_m_before=3,
    line_n_after=3
)

In [ ]:
gemini_flash_2_classification_model = GeminiModel('classify', 'models/gemini-2.0-flash', output_label_converter, True)
gemini_flash_2_classification_model.fit(classify_n_shot_dataset)
gemini_flash_2_classification_model.predict(classify_test_dataset.select(range(1)), DATASET_NAME, prompt_template,TrainStrategy.N_SHOT_TOP, 10, verbose=True)